# Albus-Hub — Deep Learning para risco de quebra de OLA/KPI

## Objetivo

Construir um MVP local para estimar, **no momento de abertura do incidente**, a probabilidade de quebra do KPI/OLA. O notebook passa pelas etapas de preparação dos dados, teste de clusterização, treinamento da ANN, avaliação do modelo e geração das previsões.

**Resumo da execução de referência:** A base contém 25.600 incidentes elegíveis e 248 violações (0.969%). No teste temporal. a ANN atingiu PR-AUC 0.0726. ROC-AUC 0.8610 e Recall 90.0%.

Como existem poucos casos positivos, Accuracy sozinha poderia dar uma visão enganosa do resultado. Por isso, usamos principalmente PR-AUC, Recall, Precision e F1, deixando ROC-AUC como métrica complementar.

## 1. Bibliotecas

As principais funções usadas no modelo foram incluídas no próprio notebook. Isso facilita acompanhar o que foi feito em cada etapa, principalmente no pré-processamento, na clusterização e na construção da ANN.

In [1]:
from collections.abc import Sequence
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import MiniBatchKMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
    silhouette_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
np.random.seed(SEED)

## 2. Contrato de dados e prevenção de leakage

Para o treinamento, consideramos apenas os incidentes com `entered_kpi_source == True`, usando `kpi_breached_source` como target. Para evitar leakage, não usamos como features informações que só aparecem depois que o incidente foi resolvido. O `closed_at` serve apenas para verificar se o resultado de um incidente anterior já era conhecido naquele momento.

In [2]:
ELIGIBILITY_COLUMN = "entered_kpi_source"

TARGET_COLUMN = "kpi_breached_source"

IDENTIFIER_COLUMNS = ["incident_id", "opened_at"]

BASE_CATEGORICAL_FEATURES = [
    "priority_code",
    "product",
    "category",
    "subcategory",
    "assigned_group",
    "configuration_item",
    "opened_by",
]

TEMPORAL_FEATURES = [
    "opened_hour",
    "opened_day_of_week",
    "opened_month",
    "is_weekend",
]

HISTORICAL_FEATURES = [
    "assigned_group_incidents_previous_1d",
    "assigned_group_incidents_previous_7d",
    "assigned_group_incidents_previous_30d",
    "assigned_group_known_outcomes_previous_30d",
    "assigned_group_breaches_previous_30d",
    "assigned_group_breach_rate_previous_30d",
    "product_incidents_previous_7d",
    "category_incidents_previous_7d",
    "priority_incidents_previous_7d",
]

CATEGORICAL_FEATURES = BASE_CATEGORICAL_FEATURES

NUMERIC_FEATURES = TEMPORAL_FEATURES + HISTORICAL_FEATURES

MODEL_FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

LEAKAGE_COLUMNS = [
    "resolved_at",
    "closed_at",
    "duration_seconds",
    "duration_hours",
    "calculated_duration_seconds",
    "duration_difference_seconds",
    "duration_mismatch",
    "closure_code",
    "solution_type",
    "status",
    ELIGIBILITY_COLUMN,
    TARGET_COLUMN,
    "entered_kpi_raw",
    "kpi_breached_raw",
    "entered_kpi_recalculated_raw",
    "kpi_breached_recalculated_raw",
    "entered_kpi_rule_mismatch",
    "kpi_breached_rule_mismatch",
]

REQUIRED_SILVER_COLUMNS = {
    "incident_id",
    "opened_at",
    "closed_at",
    "priority_code",
    "product",
    "category",
    "subcategory",
    "assigned_group",
    "configuration_item",
    "opened_by",
    ELIGIBILITY_COLUMN,
    TARGET_COLUMN,
}

class RiskDataContractError(ValueError):
    """Indica que a Silver não permite construir o modelo de risco com segurança."""

def validate_silver_for_risk(frame: pd.DataFrame, *, require_targets: bool = True) -> None:
    """Valida os requisitos mínimos da Silver para treino e inferência histórica."""
    missing = REQUIRED_SILVER_COLUMNS - set(frame.columns)
    if missing:
        raise RiskDataContractError(f"Colunas obrigatórias ausentes: {sorted(missing)}")

    if frame["incident_id"].isna().any() or frame["incident_id"].duplicated().any():
        raise RiskDataContractError("incident_id deve ser preenchido e único.")

    opened_at = pd.to_datetime(frame["opened_at"], errors="coerce")
    if opened_at.isna().any():
        raise RiskDataContractError("opened_at deve ser preenchido e válido.")

    if require_targets:
        eligible = frame[ELIGIBILITY_COLUMN].eq(True)
        if not eligible.any():
            raise RiskDataContractError("A Silver não possui incidentes elegíveis ao KPI.")

        invalid_target = eligible & frame[TARGET_COLUMN].isna()
        if invalid_target.any():
            raise RiskDataContractError("Incidentes elegíveis possuem target nulo.")

def assert_no_leakage(feature_columns: Sequence[str]) -> None:
    """Bloqueia qualquer tentativa de incluir colunas pós-desfecho no modelo."""
    forbidden = sorted(set(feature_columns) & set(LEAKAGE_COLUMNS))
    if forbidden:
        raise RiskDataContractError(f"Features proibidas por leakage: {forbidden}")

In [3]:
assert_no_leakage(MODEL_FEATURES)
pd.DataFrame({"feature_permitida": MODEL_FEATURES})

,feature_permitida
0,priority_code
1,product
2,category
3,subcategory
4,assigned_group
5,configuration_item
6,opened_by
7,opened_hour
8,opened_day_of_week
9,opened_month


## 3. Carregamento e qualidade da Silver

Cada linha da Silver representa um incidente. Antes de montar as features, verificamos duplicidade no identificador, datas de abertura inválidas e se o target está preenchido nos casos elegíveis.

In [4]:
DATA_PATH = Path("data/silver/locaweb_incidents.parquet")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset não encontrado em data/silver/locaweb_incidents.parquet. "
        "Disponibilize a Silver do projeto nesse caminho antes de executar o notebook."
    )

silver = pd.read_parquet(DATA_PATH)
validate_silver_for_risk(silver)

quality = pd.DataFrame({
    "linhas": [len(silver)],
    "colunas": [silver.shape[1]],
    "incident_id_duplicado": [int(silver["incident_id"].duplicated().sum())],
    "opened_at_nulo": [int(pd.to_datetime(silver["opened_at"], errors="coerce").isna().sum())],
    "abertura_min": [pd.to_datetime(silver["opened_at"]).min()],
    "abertura_max": [pd.to_datetime(silver["opened_at"]).max()],
})
display(quality)

nulls = silver.isna().sum().sort_values(ascending=False)
display(nulls[nulls.gt(0)].head(15).rename("nulos").to_frame())

,linhas,colunas,incident_id_duplicado,opened_at_nulo,abertura_min,abertura_max
0,122543,43,0,0,2023-01-02 20:19:58,2025-12-31 23:45:18


,nulos
parent_incident_id,107416
solution_type,107243
kpi_breached_source,96943
resolved_at,82302
closure_code,81739
product,77935
category,77721
subcategory,77720
configuration_item,1780


## 4. Feature engineering temporal e histórica

As features temporais são criadas a partir da data de abertura. Já as features históricas usam apenas acontecimentos anteriores ao incidente atual. Na taxa de quebra da equipe, por exemplo, só entram incidentes que já tinham sido encerrados naquele momento. Assim, o modelo não recebe informação do futuro.

In [5]:
def _strict_previous_counts(
    frame: pd.DataFrame,
    key: str,
    window_days: int,
) -> np.ndarray:
    """Conta aberturas anteriores na janela, excluindo timestamps simultâneos."""
    result = np.zeros(len(frame), dtype=np.int32)
    window = np.timedelta64(window_days, "D")

    for positions in frame.groupby(key, dropna=False, sort=False).indices.values():
        positions = np.asarray(positions, dtype=np.int64)
        times = frame.iloc[positions]["opened_at"].to_numpy(dtype="datetime64[ns]")
        left = np.searchsorted(times, times - window, side="left")
        right = np.searchsorted(times, times, side="left")
        result[positions] = right - left

    return result

def _known_group_outcomes_previous_30d(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Usa somente desfechos encerrados antes da abertura do incidente corrente."""
    known_count = np.zeros(len(frame), dtype=np.int32)
    breach_count = np.zeros(len(frame), dtype=np.int32)
    window = np.timedelta64(30, "D")

    outcome_events = frame.loc[
        frame[ELIGIBILITY_COLUMN].eq(True)
        & frame[TARGET_COLUMN].notna()
        & frame["closed_at"].notna(),
        ["assigned_group", "closed_at", TARGET_COLUMN],
    ].copy()
    outcome_events["closed_at"] = pd.to_datetime(outcome_events["closed_at"])

    for group, positions in frame.groupby("assigned_group", dropna=False, sort=False).indices.items():
        positions = np.asarray(positions, dtype=np.int64)
        opened = frame.iloc[positions]["opened_at"].to_numpy(dtype="datetime64[ns]")
        if pd.isna(group):
            events = outcome_events.loc[outcome_events["assigned_group"].isna()]
        else:
            events = outcome_events.loc[outcome_events["assigned_group"].eq(group)]
        events = events.sort_values("closed_at", kind="stable")
        event_times = events["closed_at"].to_numpy(dtype="datetime64[ns]")
        event_targets = events[TARGET_COLUMN].astype(np.int32).to_numpy()
        cumulative_breaches = np.concatenate(([0], np.cumsum(event_targets)))

        left = np.searchsorted(event_times, opened - window, side="left")
        right = np.searchsorted(event_times, opened, side="left")
        known_count[positions] = right - left
        breach_count[positions] = cumulative_breaches[right] - cumulative_breaches[left]

    return known_count, breach_count

def build_risk_features(
    silver: pd.DataFrame,
    *,
    require_targets: bool = True,
) -> pd.DataFrame:
    """Cria features disponíveis na abertura e históricos estritamente anteriores."""
    validate_silver_for_risk(silver, require_targets=require_targets)
    frame = silver.copy()
    frame["opened_at"] = pd.to_datetime(frame["opened_at"])
    frame["closed_at"] = pd.to_datetime(frame["closed_at"], errors="coerce")
    frame = frame.sort_values(["opened_at", "incident_id"], kind="stable").reset_index(drop=True)

    frame["opened_hour"] = frame["opened_at"].dt.hour.astype("int16")
    frame["opened_day_of_week"] = frame["opened_at"].dt.dayofweek.astype("int16")
    frame["opened_month"] = frame["opened_at"].dt.month.astype("int16")
    frame["is_weekend"] = frame["opened_day_of_week"].isin([5, 6]).astype("int8")

    for days in (1, 7, 30):
        frame[f"assigned_group_incidents_previous_{days}d"] = _strict_previous_counts(
            frame, "assigned_group", days
        )
    frame["product_incidents_previous_7d"] = _strict_previous_counts(frame, "product", 7)
    frame["category_incidents_previous_7d"] = _strict_previous_counts(frame, "category", 7)
    frame["priority_incidents_previous_7d"] = _strict_previous_counts(
        frame, "priority_code", 7
    )

    known, breached = _known_group_outcomes_previous_30d(frame)
    frame["assigned_group_known_outcomes_previous_30d"] = known
    frame["assigned_group_breaches_previous_30d"] = breached
    frame["assigned_group_breach_rate_previous_30d"] = np.divide(
        breached,
        known,
        out=np.full(len(frame), np.nan, dtype=np.float64),
        where=known > 0,
    )

    output_columns = [
        "incident_id",
        "opened_at",
        ELIGIBILITY_COLUMN,
        TARGET_COLUMN,
        *MODEL_FEATURES,
    ]
    return frame[output_columns].copy()

In [6]:
risk_features = build_risk_features(silver)
eligible = risk_features.loc[risk_features[ELIGIBILITY_COLUMN].eq(True)].copy()
eligible[TARGET_COLUMN] = eligible[TARGET_COLUMN].astype(bool)
eligible = eligible.sort_values(["opened_at", "incident_id"], kind="stable").reset_index(drop=True)

population = pd.Series({
    "total_incidents": len(silver),
    "eligible_incidents": len(eligible),
    "positive_incidents": int(eligible[TARGET_COLUMN].sum()),
    "negative_incidents": int((~eligible[TARGET_COLUMN]).sum()),
    "positive_rate": float(eligible[TARGET_COLUMN].mean()),
}, name="valor").to_frame()
display(population)

eligible[["incident_id", "opened_at", *MODEL_FEATURES, TARGET_COLUMN]].head(5)

,valor
total_incidents,122543.000000
eligible_incidents,25600.000000
positive_incidents,248.000000
negative_incidents,25352.000000
positive_rate,0.009687


,incident_id,opened_at,priority_code,product,category,subcategory,assigned_group,configuration_item,opened_by,opened_hour,...,assigned_group_incidents_previous_1d,assigned_group_incidents_previous_7d,assigned_group_incidents_previous_30d,assigned_group_known_outcomes_previous_30d,assigned_group_breaches_previous_30d,assigned_group_breach_rate_previous_30d,product_incidents_previous_7d,category_incidents_previous_7d,priority_incidents_previous_7d,kpi_breached_source
0,INC7227672,2023-01-02 20:19:58,3,lcem,cat76,sub392,Team06,IC00604,Manual,20,...,0,0,0,0,0,NaN,0,0,0,False
1,INC7236724,2023-01-09 21:23:05,3,lvps,cat139,sub290,Team14,IC09322,Manual,21,...,0,0,0,0,0,NaN,0,0,0,False
2,INC7239171,2023-01-11 15:25:46,3,lhco,cat85,sub225,Team11,IC02690,Manual,15,...,0,0,0,0,0,NaN,0,0,1,False
3,INC7239203,2023-01-11 15:54:26,3,lcem,cat76,sub392,Team06,IC00604,Manual,15,...,0,0,1,0,0,NaN,0,0,2,False
4,INC7245297,2023-01-16 16:50:07,3,lrdo,cat112,sub221,Team11,IC02001,Manual,16,...,0,1,1,0,0,NaN,0,0,3,False


## 5. Split temporal

Como os incidentes possuem ordem temporal, não usamos embaralhamento. A base é ordenada pela data de abertura e dividida em 60% treino, 20% validação e 20% teste. A validação é usada nas escolhas de modelagem, enquanto o teste fica separado até a avaliação final.

In [7]:
def split_temporally(frame, train_fraction=0.60, validation_fraction=0.20):
    train_end = int(len(frame) * train_fraction)
    validation_end = int(len(frame) * (train_fraction + validation_fraction))
    return (
        frame.iloc[:train_end].copy(),
        frame.iloc[train_end:validation_end].copy(),
        frame.iloc[validation_end:].copy(),
    )


def split_summary(frame):
    return {
        "rows": len(frame),
        "positives": int(frame[TARGET_COLUMN].sum()),
        "positive_rate": float(frame[TARGET_COLUMN].mean()),
        "opened_at_min": frame["opened_at"].min(),
        "opened_at_max": frame["opened_at"].max(),
    }

train, validation, test = split_temporally(eligible)
pd.DataFrame({
    "train": split_summary(train),
    "validation": split_summary(validation),
    "test": split_summary(test),
}).T

,rows,positives,positive_rate,opened_at_min,opened_at_max
train,15360,165,0.010742,2023-01-02 20:19:58,2025-07-26 11:08:59
validation,5120,33,0.006445,2025-07-26 11:12:55,2025-10-01 19:09:51
test,5120,50,0.009766,2025-10-01 19:09:54,2025-12-31 18:06:15


## 6. Pré-processamento para entrada da ANN

Nas variáveis numéricas, preenchemos os valores ausentes pela mediana e aplicamos `StandardScaler`. Nas categóricas, tratamos os nulos e usamos `OneHotEncoder`, permitindo categorias novas. O ajuste do pré-processamento é feito somente com o treino e depois reaplicado na validação e no teste.

In [8]:
def prepare_model_frame(frame: pd.DataFrame) -> pd.DataFrame:
    """Normaliza extensões Pandas para tipos aceitos de forma estável pelo sklearn."""
    result = frame[CATEGORICAL_FEATURES + NUMERIC_FEATURES].copy()
    for column in CATEGORICAL_FEATURES:
        result[column] = (
            result[column].astype("string").fillna("__MISSING__").astype(str)
        )
    for column in NUMERIC_FEATURES:
        result[column] = pd.to_numeric(result[column], errors="coerce").astype(float)
    return result

def build_preprocessor() -> ColumnTransformer:
    """Monta o pré-processamento reutilizado por baseline, ANN e inferência."""
    numeric = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="infrequent_if_exist",
                    max_categories=100,
                    sparse_output=False,
                    dtype=np.float32,
                ),
            ),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric, NUMERIC_FEATURES),
            ("categorical", categorical, CATEGORICAL_FEATURES),
        ],
        verbose_feature_names_out=False,
    )

In [9]:
preprocessor = build_preprocessor()
x_train = preprocessor.fit_transform(prepare_model_frame(train)).astype(np.float32)
x_validation = preprocessor.transform(prepare_model_frame(validation)).astype(np.float32)
x_test = preprocessor.transform(prepare_model_frame(test)).astype(np.float32)

y_train = train[TARGET_COLUMN].astype(np.int8).to_numpy()
y_validation = validation[TARGET_COLUMN].astype(np.int8).to_numpy()
y_test = test[TARGET_COLUMN].astype(np.int8).to_numpy()

pd.DataFrame({
    "split": ["train", "validation", "test"],
    "linhas": [len(x_train), len(x_validation), len(x_test)],
    "dimensao_transformada": [x_train.shape[1], x_validation.shape[1], x_test.shape[1]],
    "positivos": [int(y_train.sum()), int(y_validation.sum()), int(y_test.sum())],
})

,split,linhas,dimensao_transformada,positivos
0,train,15360,377,165
1,validation,5120,377,33
2,test,5120,377,50


## 7. Baseline interpretável

Uma regressão logística balanceada é usada como referência. A ANN só é interessante se produzir ganho sobre um modelo supervisionado simples no problema de classe rara.

In [10]:
def classification_metrics(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    threshold: float,
) -> dict[str, object]:
    """Calcula métricas adequadas para a classe rara de violação."""
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = probabilities >= threshold
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        predictions,
        average="binary",
        zero_division=0,
    )
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "pr_auc": float(average_precision_score(y_true, probabilities)),
        "roc_auc": float(roc_auc_score(y_true, probabilities)),
        "brier_score": float(brier_score_loss(y_true, probabilities)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "confusion_matrix": {
            "true_negative": int(tn),
            "false_positive": int(fp),
            "false_negative": int(fn),
            "true_positive": int(tp),
        },
    }

def select_operating_threshold(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    minimum_recall: float = 0.70,
) -> tuple[float, pd.DataFrame]:
    """Prioriza recall mínimo e, dentro dele, maximiza precisão e F1."""
    candidates = np.unique(
        np.concatenate(
            [
                np.arange(0.01, 0.51, 0.01),
                np.array([0.30, 0.40, 0.50, 0.60]),
                np.quantile(probabilities, np.linspace(0.70, 0.995, 40)),
            ]
        )
    )
    rows = []
    for threshold in candidates:
        metrics = classification_metrics(y_true, probabilities, float(threshold))
        rows.append(
            {
                key: metrics[key]
                for key in ["threshold", "precision", "recall", "f1"]
            }
        )
    table = pd.DataFrame(rows).sort_values("threshold").reset_index(drop=True)
    eligible = table.loc[table["recall"].ge(minimum_recall)]
    if eligible.empty:
        selected = table.sort_values(["f1", "recall", "precision"], ascending=False).iloc[0]
    else:
        selected = eligible.sort_values(
            ["precision", "f1", "threshold"], ascending=[False, False, False]
        ).iloc[0]
    return float(selected["threshold"]), table

In [11]:
baseline = LogisticRegression(
    class_weight="balanced",
    max_iter=1500,
    random_state=SEED,
    solver="lbfgs",
)
baseline.fit(x_train, y_train)
baseline_validation = baseline.predict_proba(x_validation)[:, 1]
baseline_test = baseline.predict_proba(x_test)[:, 1]

baseline_metrics = {
    "validation": classification_metrics(y_validation, baseline_validation, 0.5),
    "test": classification_metrics(y_test, baseline_test, 0.5),
}
pd.DataFrame(baseline_metrics).T

,threshold,pr_auc,roc_auc,brier_score,precision,recall,f1,confusion_matrix
validation,0.5,0.008773,0.635494,0.358406,0.010009,0.666667,0.019722,"{'true_negative': 2911, 'false_positive': 2176..."
test,0.5,0.008731,0.473061,0.655831,0.01214,0.92,0.023965,"{'true_negative': 1327, 'false_positive': 3743..."


## 8. Avaliação de clusterização

Testamos a clusterização para verificar se os grupos encontrados ajudariam na previsão. O K-Means é ajustado sem usar o target, com `k = 2, 3, 4, 5`, e o melhor valor é escolhido pelo silhouette. Depois, o cluster selecionado é adicionado a um baseline separado para comparar a PR-AUC.

Se o cluster não melhorar o resultado, ele não é usado como feature da ANN. Assim, a clusterização entra como experimento e não como uma etapa obrigatória do modelo final.

In [12]:
def evaluate_clusters(
    values: np.ndarray,
    target: np.ndarray,
    validation_values: np.ndarray,
    validation_target: np.ndarray,
    baseline_validation_pr_auc: float,
    seed: int,
) -> dict[str, object]:
    """Avalia K-Means sem usar o target no ajuste e mede separação operacional."""
    rng = np.random.default_rng(seed)
    sample_size = min(len(values), 6000)
    sample_indices = np.sort(rng.choice(len(values), size=sample_size, replace=False))
    sample_values = values[sample_indices]
    sample_target = np.asarray(target)[sample_indices]
    candidates = []

    for clusters in (2, 3, 4, 5):
        model = MiniBatchKMeans(
            n_clusters=clusters,
            random_state=seed,
            n_init=10,
            batch_size=512,
        )
        labels = model.fit_predict(sample_values)
        silhouette = silhouette_score(
            sample_values,
            labels,
            sample_size=min(2000, sample_size),
            random_state=seed,
        )
        cluster_rates = {
            str(label): float(sample_target[labels == label].mean())
            for label in range(clusters)
        }
        candidates.append(
            {
                "clusters": clusters,
                "silhouette": float(silhouette),
                "breach_rates": cluster_rates,
            }
        )

    best = max(candidates, key=lambda item: item["silhouette"])
    selected_clusters = int(best["clusters"])
    cluster_model = MiniBatchKMeans(
        n_clusters=selected_clusters,
        random_state=seed,
        n_init=10,
        batch_size=512,
    ).fit(values)
    train_labels = cluster_model.predict(values)
    validation_labels = cluster_model.predict(validation_values)
    identity = np.eye(selected_clusters, dtype=np.float32)
    augmented_train = np.column_stack([values, identity[train_labels]])
    augmented_validation = np.column_stack(
        [validation_values, identity[validation_labels]]
    )
    comparison_model = LogisticRegression(
        class_weight="balanced",
        max_iter=1500,
        random_state=seed,
        solver="lbfgs",
    ).fit(augmented_train, target)
    augmented_probability = comparison_model.predict_proba(augmented_validation)[:, 1]
    augmented_pr_auc = float(
        average_precision_score(validation_target, augmented_probability)
    )
    absolute_gain = augmented_pr_auc - baseline_validation_pr_auc
    useful = best["silhouette"] >= 0.10 and absolute_gain >= 0.005
    return {
        "sample_size": sample_size,
        "candidates": candidates,
        "selected_clusters": selected_clusters,
        "selected_silhouette": best["silhouette"],
        "baseline_validation_pr_auc": baseline_validation_pr_auc,
        "baseline_with_cluster_validation_pr_auc": augmented_pr_auc,
        "absolute_pr_auc_gain": absolute_gain,
        "use_as_model_feature": useful,
        "conclusion": (
            "Cluster melhora materialmente o baseline e deve ser testado no modelo final."
            if useful
            else "Cluster não melhora o PR-AUC o suficiente para entrar no modelo final."
        ),
    }

In [13]:
cluster_results = evaluate_clusters(
    x_train,
    y_train,
    x_validation,
    y_validation,
    float(average_precision_score(y_validation, baseline_validation)),
    SEED,
)

display(pd.DataFrame(cluster_results["candidates"]))
pd.Series({
    "k_selecionado": cluster_results["selected_clusters"],
    "silhouette": cluster_results["selected_silhouette"],
    "PR_AUC_baseline": cluster_results["baseline_validation_pr_auc"],
    "PR_AUC_com_cluster": cluster_results["baseline_with_cluster_validation_pr_auc"],
    "ganho_absoluto": cluster_results["absolute_pr_auc_gain"],
    "usar_cluster_como_feature": cluster_results["use_as_model_feature"],
    "conclusao": cluster_results["conclusion"],
}, name="resultado").to_frame()

,clusters,silhouette,breach_rates
0,2,0.172303,"{'0': 0.00707361802462667, '1': 0.017865322950..."
1,3,0.147218,"{'0': 0.0032804811372334607, '1': 0.0099403578..."
2,4,0.148089,"{'0': 0.011124845488257108, '1': 0.00269360269..."
3,5,0.112776,"{'0': 0.02967741935483871, '1': 0.007866273352..."


,resultado
k_selecionado,2
silhouette,0.172303
PR_AUC_baseline,0.008773
PR_AUC_com_cluster,0.00876
ganho_absoluto,-0.000013
usar_cluster_como_feature,False
conclusao,Cluster não melhora o PR-AUC o suficiente para...


## 9. Construção e parametrização da ANN

Foram testadas duas arquiteturas densas. Cada camada oculta usa ReLU seguida de Dropout, enquanto a saída possui um neurônio com Sigmoid. O treinamento usa Adam, binary cross-entropy e PR-AUC como métrica de validação.

Como a classe positiva é rara, usamos `class_weight`. O `EarlyStopping` recupera os melhores pesos e o `ReduceLROnPlateau` reduz a taxa de aprendizado quando a PR-AUC de validação para de melhorar.

In [14]:
@dataclass(frozen=True)
class ANNConfig:
    """Configuração compacta de uma ANN binária."""

    name: str
    hidden_units: tuple[int, ...]
    dropout: float
    learning_rate: float
    batch_size: int = 256
    epochs: int = 60
    patience: int = 7

    def to_dict(self) -> dict[str, object]:
        return asdict(self)

ANN_CONFIGS = [
    ANNConfig(
        name="compact-64-32",
        hidden_units=(64, 32),
        dropout=0.25,
        learning_rate=0.001,
    ),
    ANNConfig(
        name="reference-128-64-32",
        hidden_units=(128, 64, 32),
        dropout=0.30,
        learning_rate=0.001,
    ),
]

def build_ann(
    input_dimension: int,
    config: ANNConfig,
    *,
    compile_model: bool = True,
):
    """Constrói a rede Dense/ReLU/Dropout com saída sigmoide."""
    import tensorflow as tf

    layers = [tf.keras.layers.Input(shape=(input_dimension,))]
    for units in config.hidden_units:
        layers.extend(
            [
                tf.keras.layers.Dense(units, activation="relu"),
                tf.keras.layers.Dropout(config.dropout),
            ]
        )
    layers.append(tf.keras.layers.Dense(1, activation="sigmoid"))
    model = tf.keras.Sequential(layers, name=f"risk_{config.name}")
    if compile_model:
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=config.learning_rate),
            loss="binary_crossentropy",
            metrics=[tf.keras.metrics.AUC(curve="PR", name="pr_auc")],
        )
    return model

def train_ann(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_validation: np.ndarray,
    y_validation: np.ndarray,
    config: ANNConfig,
    seed: int,
):
    """Treina uma configuração com pesos de classe e early stopping."""
    import tensorflow as tf

    tf.keras.utils.set_random_seed(seed)
    model = build_ann(x_train.shape[1], config)
    positives = max(int(np.asarray(y_train).sum()), 1)
    negatives = max(len(y_train) - positives, 1)
    class_weight = {0: 1.0, 1: negatives / positives}
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_pr_auc",
            mode="max",
            patience=config.patience,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_pr_auc",
            mode="max",
            factor=0.5,
            patience=max(2, config.patience // 2),
            min_lr=1e-5,
        ),
    ]
    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_validation, y_validation),
        epochs=config.epochs,
        batch_size=config.batch_size,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=0,
    )
    return model, history.history, class_weight

def predict_ann(model, values: np.ndarray) -> np.ndarray:
    """Retorna a probabilidade sigmoide em um vetor unidimensional."""
    return model.predict(values, batch_size=1024, verbose=0).reshape(-1)

In [15]:
pd.DataFrame([config.to_dict() for config in ANN_CONFIGS])

,name,hidden_units,dropout,learning_rate,batch_size,epochs,patience
0,compact-64-32,"(64, 32)",0.25,0.001,256,60,7
1,reference-128-64-32,"(128, 64, 32)",0.30,0.001,256,60,7


## 10. Treinamento e seleção da arquitetura

A arquitetura é escolhida **somente pela PR-AUC da validação**. O conjunto de teste não participa da escolha.

In [16]:
ann_results = []
trained_models = {}

for index, ann_config in enumerate(ANN_CONFIGS):
    model, history, class_weight = train_ann(
        x_train,
        y_train,
        x_validation,
        y_validation,
        ann_config,
        seed=SEED + index,
    )
    validation_probability = predict_ann(model, x_validation)
    test_probability = predict_ann(model, x_test)
    validation_pr_auc = float(average_precision_score(y_validation, validation_probability))

    ann_results.append({
        "config": ann_config.to_dict(),
        "epochs_run": len(history["loss"]),
        "class_weight": class_weight,
        "validation_pr_auc": validation_pr_auc,
        "validation_metrics_at_0_5": classification_metrics(
            y_validation, validation_probability, 0.5
        ),
    })
    trained_models[ann_config.name] = (
        model,
        validation_probability,
        test_probability,
    )

candidate_table = pd.DataFrame([
    {
        **item["config"],
        "epochs_run": item["epochs_run"],
        "class_weight_positive": item["class_weight"][1],
        "validation_pr_auc": item["validation_pr_auc"],
    }
    for item in ann_results
])
display(candidate_table)

selected_result = max(ann_results, key=lambda item: item["validation_pr_auc"])
selected_name = selected_result["config"]["name"]
selected_model, ann_validation_raw, ann_test_raw = trained_models[selected_name]
print(f"ANN selecionada: {selected_name}")
selected_model.summary()

,name,hidden_units,dropout,learning_rate,batch_size,epochs,patience,epochs_run,class_weight_positive,validation_pr_auc
0,compact-64-32,"(64, 32)",0.25,0.001,256,60,7,14,92.090909,0.058187
1,reference-128-64-32,"(128, 64, 32)",0.30,0.001,256,60,7,10,92.090909,0.040913


ANN selecionada: compact-64-32


Model: "risk_compact-64-32"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │        24,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 78,917 (308.27 KB)

 Trainable params: 26,305 (102.75 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 52,612 (205.52 KB)

## 11. Calibração e escolha do threshold

A saída Sigmoid é calibrada com Platt scaling usando a validação. O threshold também é escolhido nessa etapa: buscamos Recall de pelo menos 70% e, entre os thresholds que atendem esse critério, priorizamos melhores valores de Precision/F1. Depois disso, o threshold escolhido é aplicado ao teste final.

In [17]:
def probability_logit(probabilities: np.ndarray) -> np.ndarray:
    """Transforma probabilidades em logits com proteção numérica."""
    clipped = np.clip(np.asarray(probabilities, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(clipped / (1 - clipped)).reshape(-1, 1)

def fit_probability_calibrator(
    probabilities: np.ndarray,
    target: np.ndarray,
) -> LogisticRegression:
    """Ajusta calibração de Platt exclusivamente na janela de validação."""
    calibrator = LogisticRegression(random_state=42)
    calibrator.fit(probability_logit(probabilities), np.asarray(target, dtype=int))
    return calibrator

def apply_probability_calibrator(calibrator, probabilities: np.ndarray) -> np.ndarray:
    """Aplica a calibração salva sem reestimar parâmetros."""
    return calibrator.predict_proba(probability_logit(probabilities))[:, 1]

In [18]:
calibrator = fit_probability_calibrator(ann_validation_raw, y_validation)
ann_validation = apply_probability_calibrator(calibrator, ann_validation_raw)
ann_test = apply_probability_calibrator(calibrator, ann_test_raw)

selected_threshold, threshold_table = select_operating_threshold(
    y_validation,
    ann_validation,
    minimum_recall=0.70,
)
print(f"Threshold selecionado: {selected_threshold:.6f}")
threshold_table.sort_values(["recall", "precision"], ascending=False).head(10)

Threshold selecionado: 0.007790


,threshold,precision,recall,f1
1,0.007790,0.017368,0.787879,0.033987
0,0.007578,0.016927,0.787879,0.033142
3,0.008252,0.016901,0.727273,0.033035
2,0.008043,0.016450,0.727273,0.032172
4,0.008522,0.015930,0.666667,0.031117
6,0.009054,0.016104,0.636364,0.031414
5,0.008849,0.015637,0.636364,0.030523
8,0.009506,0.016313,0.606061,0.031771
7,0.009300,0.015810,0.606061,0.030817
12,0.010368,0.017117,0.575758,0.033246


## 12. Avaliação de desempenho

Como há poucos casos de quebra do KPI/OLA, damos mais atenção à PR-AUC e ao Recall. Também analisamos ROC-AUC, Precision, F1 e a matriz de confusão para entender quantas quebras o modelo consegue encontrar e quantos falsos alertas ele gera com o threshold escolhido.

In [19]:
ann_validation_metrics = classification_metrics(
    y_validation, ann_validation, selected_threshold
)
ann_test_metrics = classification_metrics(y_test, ann_test, selected_threshold)

comparison = pd.DataFrame({
    "baseline_test@0.5": baseline_metrics["test"],
    "ann_validation": ann_validation_metrics,
    "ann_test": ann_test_metrics,
}).T

display(comparison[["threshold", "pr_auc", "roc_auc", "brier_score", "precision", "recall", "f1"]])

pd.Series(ann_test_metrics["confusion_matrix"], name="quantidade").to_frame()

,threshold,pr_auc,roc_auc,brier_score,precision,recall,f1
baseline_test@0.5,0.5,0.008731,0.473061,0.655831,0.01214,0.92,0.023965
ann_validation,0.00779,0.058187,0.808579,0.006318,0.017368,0.787879,0.033987
ann_test,0.00779,0.072555,0.860982,0.009496,0.02275,0.9,0.044379


,quantidade
true_negative,3137
false_positive,1933
false_negative,5
true_positive,45


## 13. Previsões reais

Na tabela abaixo aplicamos a ANN escolhida aos incidentes do conjunto de teste. A coluna `breach_probability` mostra a probabilidade calibrada de quebra, enquanto `predicted_breach` indica o resultado após aplicar o threshold escolhido na validação.

In [20]:
predictions = test[["incident_id", "opened_at", "priority_code", TARGET_COLUMN]].copy()
predictions["breach_probability"] = ann_test
predictions["predicted_breach"] = ann_test >= selected_threshold

predictions.sort_values("breach_probability", ascending=False).head(15)

,incident_id,opened_at,priority_code,kpi_breached_source,breach_probability,predicted_breach
25544,INC8650448,2025-12-28 08:09:00,3,False,0.355911,True
25531,INC8649414,2025-12-27 11:10:46,3,False,0.314069,True
25540,INC8649768,2025-12-27 21:59:04,3,True,0.284371,True
25475,INC8646934,2025-12-25 03:00:54,3,False,0.276905,True
25566,INC8651433,2025-12-29 10:00:09,3,False,0.273762,True
25494,INC8647636,2025-12-26 00:16:31,3,False,0.259835,True
25541,INC8649800,2025-12-27 22:22:46,3,True,0.251360,True
25453,INC8646280,2025-12-24 12:31:55,3,False,0.241011,True
25560,INC8651368,2025-12-29 08:21:44,3,False,0.240814,True
25513,INC8648147,2025-12-26 12:37:33,3,False,0.228545,True


## 14. Interpretação e limitações

- A classe positiva é rara. Por isso, aumentar o Recall também tende a aumentar a quantidade de falsos positivos. O modelo deve apoiar **priorização**, não decisão automática.
- O split temporal foi mantido para respeitar a ordem dos incidentes e reduzir o risco de leakage.
- A clusterização só entra no modelo se realmente melhorar a PR-AUC; separar perfis por si só não é suficiente.
- A calibração foi feita apenas com a validação e precisa ser revista quando houver novos dados.
- Em uma aplicação futura, também será necessário acompanhar mudanças no comportamento dos dados e revisar periodicamente o threshold e a calibração.

## Conclusão

Os testes mostram que é possível usar os dados disponíveis na abertura do incidente para estimar o risco de quebra do KPI/OLA. Durante o desenvolvimento, comparamos diferentes configurações da ANN e também testamos se a clusterização ajudaria no resultado.

A avaliação respeitou a ordem temporal dos dados e manteve o conjunto de teste separado das decisões de modelagem. Como as violações são raras, o principal ponto de atenção continua sendo o equilíbrio entre identificar as quebras e evitar um número excessivo de falsos alertas.